In [1]:
pip install matplotlib pandas numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
RQ3 CHARTS — 3 charts for slides
══════════════════════════════════
Chart 1 → RAGAS grouped bar chart
          Standard vs Balanced prompt
          (Faithfulness + Answer Relevancy side by side)
          Source: ragas_comparison_table.csv

Chart 2 → NDCG vs SPD Pareto curve (RQ3)
          Shows relevance-fairness trade-off across methods
          Source: comparison_table.csv

Chart 3 → SPD bar chart across all lambda values
          Shows how MMR affects privilege bias
          Source: comparison_table.csv

OUTPUT:
  rq3_charts.png  (all 3 charts in one figure)
"""

import matplotlib
matplotlib.use("Agg")   # works in Colab + local
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import os

# ── Paths — edit if your files are elsewhere ──────────────────
COMPARISON_FILE  = "comparison_table.csv"
RAGAS_FILE       = "ragas_comparison_table.csv"
OUTPUT_FILE      = "rq3_charts.png"

# ── Color palette ─────────────────────────────────────────────
COL_BASELINE  = "#636EFA"
COL_MMR       = ["#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
COL_STANDARD  = "#4C72B0"
COL_BALANCED  = "#55A868"
COL_POSITIVE  = "#55A868"
COL_NEGATIVE  = "#C44E52"
COL_NEUTRAL   = "#636EFA"


# ══════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════
print("Loading data ...")

# comparison_table.csv
comp_df = pd.read_csv(COMPARISON_FILE)
print(f"  comparison_table.csv: {len(comp_df)} rows")
print(f"  Columns: {list(comp_df.columns)}")

# ragas_comparison_table.csv
ragas_df = pd.read_csv(RAGAS_FILE)
print(f"  ragas_comparison_table.csv: {len(ragas_df)} rows")
print(f"  Columns: {list(ragas_df.columns)}")


# ══════════════════════════════════════════════════════════════
# FIGURE SETUP — 1 row × 3 charts
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("RQ3 — MMR Reranking: Fairness vs Utility Analysis",
             fontsize=15, fontweight="bold", y=1.02)

plt.subplots_adjust(wspace=0.35)


# ══════════════════════════════════════════════════════════════
# CHART 1 — RAGAS Grouped Bar Chart
# Standard vs Balanced Prompt
# Faithfulness + Answer Relevancy side by side
# ══════════════════════════════════════════════════════════════
ax1 = axes[0]

# extract values from ragas_comparison_table.csv
# expected columns: Metric, Standard Prompt, Balanced Prompt
ragas_metrics = ragas_df[ragas_df["Metric"].isin(["Faithfulness", "Answer Relevancy"])]

metrics      = ragas_metrics["Metric"].tolist()
std_scores   = ragas_metrics["Standard Prompt"].tolist()
bal_scores   = ragas_metrics["Balanced Prompt"].tolist()

x      = np.arange(len(metrics))
width  = 0.35

bars1 = ax1.bar(x - width/2, std_scores, width,
                label="Standard Prompt",
                color=COL_STANDARD, alpha=0.85, edgecolor="white", linewidth=0.8)
bars2 = ax1.bar(x + width/2, bal_scores, width,
                label="Balanced Prompt",
                color=COL_BALANCED, alpha=0.85, edgecolor="white", linewidth=0.8)

# value labels on bars
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f"{bar.get_height():.3f}",
             ha="center", va="bottom", fontsize=9, fontweight="bold",
             color=COL_STANDARD)

for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f"{bar.get_height():.3f}",
             ha="center", va="bottom", fontsize=9, fontweight="bold",
             color=COL_BALANCED)

ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontsize=11)
ax1.set_ylabel("Score", fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_title("Chart 1 — RAGAS: Standard vs Balanced Prompt",
              fontsize=11, fontweight="bold", pad=10)
ax1.legend(fontsize=9, loc="lower right")
ax1.grid(axis="y", alpha=0.3, linestyle="--")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# annotation showing improvement
for i, (s, b) in enumerate(zip(std_scores, bal_scores)):
    diff = b - s
    color = COL_POSITIVE if diff > 0 else COL_NEGATIVE
    ax1.annotate(f"{'+'if diff>0 else ''}{diff:.3f}",
                 xy=(x[i] + width/2, b + 0.03),
                 fontsize=8, ha="center", color=color, fontstyle="italic")


# ══════════════════════════════════════════════════════════════
# CHART 2 — NDCG vs SPD Pareto Curve (RQ3)
# X-axis = SPD (lower = fairer)
# Y-axis = NDCG@10 (higher = more relevant)
# Ideal point = bottom-left (low SPD, high NDCG)
# ══════════════════════════════════════════════════════════════
ax2 = axes[1]

methods = comp_df["method"].tolist()
ndcg    = comp_df["NDCG_at_10"].tolist()
spd     = comp_df["SPD"].tolist()

# color each point
colors = [COL_BASELINE] + COL_MMR[:len(methods)-1]

# plot points
for i, (m, n, s, c) in enumerate(zip(methods, ndcg, spd, colors)):
    ax2.scatter(s, n, color=c, s=120, zorder=5,
                edgecolors="white", linewidths=1.5)
    # label offset to avoid overlap
    offset_x = 0.001
    offset_y = 0.003
    if m == "Baseline":
        offset_x = 0.001
        offset_y = 0.005
    ax2.annotate(m,
                 xy=(s, n),
                 xytext=(s + offset_x, n + offset_y),
                 fontsize=8, fontweight="bold", color=c,
                 arrowprops=dict(arrowstyle="-", color=c, lw=0.8))

# connect points with dashed line (Pareto frontier)
# sort by SPD for the curve
sorted_pairs = sorted(zip(spd, ndcg, methods, colors))
spd_sorted   = [p[0] for p in sorted_pairs]
ndcg_sorted  = [p[1] for p in sorted_pairs]
ax2.plot(spd_sorted, ndcg_sorted, "--", color="grey",
         alpha=0.5, linewidth=1.2, zorder=1)

# shade the "ideal zone" (low SPD, high NDCG)
ax2.axhline(y=np.mean(ndcg), color="grey", linestyle=":",
            linewidth=0.8, alpha=0.5, label=f"Avg NDCG={np.mean(ndcg):.3f}")
ax2.axvline(x=0, color="grey", linestyle=":",
            linewidth=0.8, alpha=0.5, label="SPD=0 (fair)")

ax2.set_xlabel("SPD — Statistical Parity Difference\n(← fairer)", fontsize=10)
ax2.set_ylabel("NDCG@10  (higher = more relevant →)", fontsize=10)
ax2.set_title("Chart 2 — NDCG vs SPD Pareto Curve (RQ3)",
              fontsize=11, fontweight="bold", pad=10)
ax2.legend(fontsize=8, loc="lower right")
ax2.grid(alpha=0.3, linestyle="--")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# ideal corner annotation
ax2.annotate("← Ideal Zone\n(fair + relevant)",
             xy=(min(spd), max(ndcg)),
             fontsize=8, color="green", alpha=0.7,
             ha="left", va="top")


# ══════════════════════════════════════════════════════════════
# CHART 3 — SPD Bar Chart across all Lambda Values
# Shows how MMR λ affects privilege bias
# Baseline shown as reference line
# ══════════════════════════════════════════════════════════════
ax3 = axes[2]

# separate baseline from MMR
baseline_spd = comp_df[comp_df["method"] == "Baseline"]["SPD"].values[0]
mmr_df       = comp_df[comp_df["method"] != "Baseline"].copy()
mmr_labels   = mmr_df["method"].tolist()
mmr_spd      = mmr_df["SPD"].tolist()

# color bars by SPD vs baseline
bar_colors = [
    COL_POSITIVE if s < baseline_spd else COL_NEGATIVE
    for s in mmr_spd
]

bars = ax3.bar(mmr_labels, mmr_spd,
               color=bar_colors, alpha=0.85,
               edgecolor="white", linewidth=0.8)

# value labels
for bar, val in zip(bars, mmr_spd):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.0005,
             f"{val:.3f}",
             ha="center", va="bottom",
             fontsize=10, fontweight="bold")

# baseline reference line
ax3.axhline(y=baseline_spd, color=COL_BASELINE,
            linestyle="--", linewidth=1.5,
            label=f"Baseline SPD = {baseline_spd:.3f}")

# annotation on baseline line
ax3.annotate(f"Baseline = {baseline_spd:.3f}",
             xy=(len(mmr_labels)-1, baseline_spd),
             xytext=(len(mmr_labels)-0.5, baseline_spd + 0.002),
             fontsize=8, color=COL_BASELINE, fontweight="bold")

ax3.set_xlabel("MMR Lambda (λ)", fontsize=11)
ax3.set_ylabel("SPD — Statistical Parity Difference", fontsize=11)
ax3.set_title("Chart 3 — SPD across Lambda Values\n(green = fairer than baseline)",
              fontsize=11, fontweight="bold", pad=10)
ax3.legend(fontsize=9)
ax3.grid(axis="y", alpha=0.3, linestyle="--")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

# legend patches
green_patch = mpatches.Patch(color=COL_POSITIVE, alpha=0.85,
                              label="Fairer than baseline")
red_patch   = mpatches.Patch(color=COL_NEGATIVE, alpha=0.85,
                              label="Less fair than baseline")
ax3.legend(handles=[green_patch, red_patch,
                    mpatches.Patch(color=COL_BASELINE, label=f"Baseline={baseline_spd:.3f}")],
           fontsize=8, loc="upper left")


# ══════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════
plt.tight_layout()
plt.savefig(OUTPUT_FILE, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nCharts saved to: {OUTPUT_FILE}")

# show in Colab/Jupyter
try:
    from IPython.display import Image, display
    display(Image(filename=OUTPUT_FILE))
except Exception:
    pass